# Opposite double gyres and a linear/nonlinear wind ramp

This notebook calls two independent comparisons:

- **Patches:** spin up original and reversed winds independently; add the same balanced patch to each, with an evolving control for each wind sign.
- **Wind ramp:** start nearly from rest and apply the same increasing wind in a genuinely linear model and the full nonlinear model. Broad double-gyre wind forcing forms a western-intensified circulation. The current default is a moderate adjustment/recirculation comparison, not a validated instability threshold. The two-panel movie reports the instantaneous peak wind stress.

The patch and ramp experiments have separate settings. Select `args.part = "patches"` or `"wind-ramp"` to run only one.

**Predict:** How does current reversal change the patch response? At which wind strengths does nonlinear evolution depart from the linear approximation? Does variability persist while the maximum wind is held fixed?

## Run the experiment

This notebook calls the corresponding script, so there is one implementation to maintain. Full runs integrate many simulated days and may take several minutes. GIF encoding adds time. Generated outputs should not be committed.

In [ ]:
from pathlib import Path
import runpy
import sys

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / "src" / "shallowwater").is_dir() and (p / "scripts").is_dir())
if str(ROOT / "scripts") not in sys.path:
    sys.path.insert(0, str(ROOT / "scripts"))
experiment = runpy.run_path(str(ROOT / "scripts" / "14_double_gyre_adjustment.py"))
args = experiment["make_parser"]().parse_args(["--quick", "--frames", "120", "--part", "wind-ramp"])
# Optional adjustments before running:
# args.smoke = True                 # short workflow check
# args.quick = False                # default higher resolution
# args.drag_days = 30.0
# args.patch_amplitude = 0.10
# args.part = "patches"             # or "all" for both comparisons
# args.ramp_tau_max = 0.04
# args.ramp_field = "vorticity"
# args.increase_wind = True         # continue saved full-resolution baseline
# args.wind_levels = [.04, .06, .08, .10]
# args.wind_stage_days = 60.
# args.wind_hold_days = 120.
# args.wind_drag_days = 80.         # optional; applied to both models
# args.wind_viscosity = 500.        # optional; applied to both models
# args.replot = True                # redraw saved fields without integration
if args.replot:
    experiment["replot_saved"](args)
else:
    results = experiment["run_experiment"](args)

## Inspect and interpret

The patch animation has the eastward-jet case on top and the westward case below; total height is on the left and departures from the corresponding control on the right. Black contours show that row's evolving control. Each column has a shared fixed scale.

The wind-ramp animation compares linear and full shallow-water dynamics under identical broad wind stress. The current defaults use a 2000 × 1600 km basin, beta 2×10⁻¹¹ m⁻¹ s⁻¹, 40-day drag and uniform viscosity of 1000 m²/s. Stress rises from zero to 0.04 N/m² over 120 days, followed by 120 days at constant stress. The default grid is 128 × 96; quick mode is 96 × 64.

Check the western/eastern boundary-current ratio in the diagnostics as well as energy and variability. Western intensification requires a ratio above one. The earlier narrow-band, weak-beta example developed an overdriven eastern recirculation; it is archived. Its numerical checks did not validate the intended gyre regime.

The revised default is nearly steady. It should not be labelled a barotropic-instability threshold; experiment 13 remains the controlled instability illustration. Changing wind or dissipation requires checking the circulation regime again.

Both models start close to rest without an imposed reference flow. The cache permits redrawing with height or vorticity. Set `args.replot = True` and choose `args.part` to redraw saved results. Cache matching includes physical and numerical settings; mixing/forcing fingerprints also cover the experiment-specific operators.

Enhanced coastal viscosity is optional. The default has the same viscosity at the coast and in the interior. Its additional variable-viscosity operator is tested to dissipate, rather than generate, kinetic energy.

In [ ]:
from IPython.display import Image, display
names = []
if args.part != "wind-ramp":
    names += ["eta_14_balanced_patch.gif", "experiment_14_wave_propagation.png"]
if args.part != "patches":
    names += ["eta_14_wind_step.gif", "experiment_14_wind_ramp_diagnostics.png"]
for name in names:
    display(Image(filename=str(args.output_dir / name)))